# AKT — kiểm CUDA tổng hợp có giới hạn

Chọn Runtime → Change runtime type → GPU. Upload bundle source được tạo bằng `scripts/akt/build_colab_bundle.py`. Chạy tuần tự các cells và tải kết quả trước khi runtime dừng.

Dữ liệu tổng hợp400targets, một optimizer step. Đây là kiểm tương thích thiết bị và artifact lifecycle. Resume giữa epoch và Gate4 vẫn cần kiểm riêng. Nguồn và giới hạn trong `docs/akt-colab-synthetic-runbook.md`.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, io, uuid, os, sys, subprocess, importlib.util
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Chỉ upload một ZIP source bundle')
name, data = next(iter(uploaded.items()))
if len(data) > 20 * 1024 * 1024:
    raise ValueError('Bundle vượt20MiB')
ROOT = Path('/content') / ('kltn-synthetic-' + uuid.uuid4().hex)
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)) or sum(x.file_size for x in archive.infolist()) > 50 * 1024 * 1024:
        raise ValueError('Archive duplicate hoặc vượt50MiB expanded')
    manifest = json.loads(archive.read('BUNDLE_MANIFEST.json'))
    if (manifest.get('schema_version'), manifest.get('scope')) not in {(1,'source_and_synthetic_tests_only'),(2,'source_and_bounded_test_fixtures')}:
        raise ValueError('Sai bundle schema/scope')
    if set(names) != set(manifest['files']) | {'BUNDLE_MANIFEST.json'}:
        raise ValueError('Bundle thiếu/thừa file')
    for path, expected in manifest['files'].items():
        relative = PurePosixPath(path)
        if relative.is_absolute() or '..' in relative.parts or '\\' in path or ':' in path:
            raise ValueError('Unsafe archive path')
        content = archive.read(path)
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Hash mismatch ' + path)
        destination = ROOT.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)
(ROOT / 'BUNDLE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
(ROOT / 'BUNDLE_SHA256.txt').write_text(hashlib.sha256(data).hexdigest() + '\n')
print('Source snapshot', ROOT, 'SHA256', hashlib.sha256(data).hexdigest())


In [ ]:
# Venv riêng dưới project, kế thừa PyTorch runtime; chỉ cài thiếu vào venv.
subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(ROOT / '.venv')],check=True,timeout=120)
AKT_PY = str(ROOT / '.venv/bin/python')
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Runtime chưa có PyTorch, cần chuẩn bị image trước')
packages = {'numpy':'numpy', 'pandas':'pandas', 'duckdb':'duckdb',
            'sklearn':'scikit-learn', 'psutil':'psutil', 'pytest':'pytest', 'networkx':'networkx'}
probe = 'import importlib.util,json; print(json.dumps([p for m,p in ' + repr(packages) + '.items() if importlib.util.find_spec(m) is None]))'
missing = json.loads(subprocess.check_output([AKT_PY,'-c',probe],text=True))
if missing:
    subprocess.run([AKT_PY, '-m', 'pip', 'install', *missing], check=True, timeout=300)
upstream = ROOT / 'artifacts/akt/dependencies/upstream'
upstream.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(['git','clone','--no-checkout','https://github.com/arghosh/AKT.git',str(upstream)],check=True,timeout=180)
subprocess.run(['git','-C',str(upstream),'checkout','--detach','01b7e8d7d7c862cabc9467108e3e7f2555653a88'],check=True,timeout=60)
ENV = dict(os.environ, PYTHONPATH=str(ROOT / 'src'), PYTHONDONTWRITEBYTECODE='1')
OUTPUT = ROOT / 'colab-evidence'
OUTPUT.mkdir()
(OUTPUT / 'BUNDLE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
(OUTPUT / 'BUNDLE_SHA256.txt').write_text(hashlib.sha256(data).hexdigest() + '\n')


In [ ]:
# CPU suite trong subprocess riêng, timeout5phút. Lỗi thì dừng trước GPU probe.
with (OUTPUT / 'cpu-tests.log').open('w') as log:
    result = subprocess.run([AKT_PY,'-B','-m','pytest','tests/akt','tests/shared_kt','tests/gam_kt',
                             '-q','-p','no:cacheprovider','-k','not test_native_akt_import_cpu and not test_actual_frozen_assignments_are_unchanged_and_student_disjoint','--basetemp',str(ROOT / 'pytest-temp')],
                            cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES=''),stdout=log,stderr=subprocess.STDOUT,timeout=300)
print((OUTPUT / 'cpu-tests.log').read_text()[-6000:])
result.check_returncode()


In [ ]:
# Một synthetic epoch gồm một batch,400targets, timeout10phút.
# Probe từ chối device cuda nếu GPU không có, không tự fallback CPU.
RUN = ROOT / 'experiments/akt/results/colab-synthetic-cuda'
with (OUTPUT / 'cuda-probe.log').open('w') as log:
    result = subprocess.run([AKT_PY,'-B','scripts/akt/synthetic_device_probe.py',
                             '--device','cuda','--output-dir',str(RUN)],
                            cwd=ROOT,env=ENV,stdout=log,stderr=subprocess.STDOUT,timeout=600)
print((OUTPUT / 'cuda-probe.log').read_text()[-6000:])
result.check_returncode()
report = json.loads((RUN / 'report.json').read_text())
assert report['status'] == 'passed_synthetic' and report['cuda_lifecycle']
print({key:report[key] for key in ['status','gpu_name','peak_vram_bytes','training_seconds','gate4_pass','mid_epoch_resume_verified']})


In [ ]:
# Cell này vẫn có thể chạy thủ công khi test/probe fail để tải diagnostic logs.
import shutil
from google.colab import files
if 'RUN' in globals() and RUN.exists():
    shutil.copytree(RUN, OUTPUT / 'synthetic-run', dirs_exist_ok=True)
archive_path = shutil.make_archive(str(ROOT / 'akt-colab-results'), 'zip', OUTPUT)
files.download(archive_path)
